In [1]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import numpy as np
import matplotlib as mpl
import plotconfig
from matplotlib.lines import Line2D

In [2]:
args = {
    "chaos": "on_1",
    "qdisc": ["fq", ""],
    "timestamp": ["202602", "202602", "202511", "202602"],
    "cca": ["cubic"],
    "test_cca": "yes",
    "kernel": ["kernel6-1"],

    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [100],
    "delay_rtt": [10,20,30,40],
    "deadline_run": [1000000, 10000000, 20000000],
}

metric = "bits_per_second"

baselogpath = "../data"

In [3]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 4590
end 4590
bytes 4590
bits_per_second 4590
mbps_timeseries 4590
rttms_timeseries 4590
retransmits 4590
timestamp 4590
iteration 4590
cpu_host_total 4590
cpu_host_user 4590
cpu_host_system 4590
cpu_remote_total 4590
chaos 4590
deadline_run 4590
deadline_period 4590
os 4590
bdp 4590
setup 4590
cca 4590
cpus 4590
kernel 4590
mode 4590
loss 4590
rate 4590
delay_rtt 4590
buffer_size_bytes 4590
parallel 4590
socket_buffer 4590
app_buffer 4590
n 4590
sysctl_cmd 4590
vm 4590
bandwidth_delay_product 4590
loss_mode 4590
vms 4590
pacing 4590
hyperthreading 4590
tso 4590
qdisc 4590
hpet 4590
tsc 4590
hostq 4590
loadperc 4590
deadline_period_factor 4590
random_loss_rate 4590
gemodel_q 4590
original_cca 4590
test_cca 4590
default_qdisc 4590
json 4590


In [4]:
df["slice_perc"] = round((df["deadline_run"]/df["deadline_period"])*100)
df["cca_generic"] = df["cca"] + df["qdisc"]
df["patch_or_not"] = df["cca"].apply(lambda x: "patched" if "-patched" in x else "original")
df["mbps"] = df["bits_per_second"]/1000000

In [5]:
def strip(df, savefig = False):    
    if savefig:
        mpl.use('agg')
    plotconfig.configure_conext()
    height = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)*(1/3)
    width = height *(1.4)
    FIG_SIZE = (width, height)
    
    paletti = plotconfig.COLORS[:3]
    fig, axs = plt.subplots(1,1,figsize=FIG_SIZE, sharey=True,constrained_layout=True)


    for it, cca in enumerate(sorted(df["cca_generic"].unique())):
        if cca == "cubic":
            df_cubic = df[df["cca_generic"] == "cubic"]
            df_cubic = df_cubic.sort_values(by=['slice_perc', "delay_rtt"])
            df_cubic = df_cubic[["slice_perc", "mbps","deadline_run", "delay_rtt"]].groupby(["slice_perc", "deadline_run", "delay_rtt"],as_index=False).median()
            for rtt_ in df_cubic["delay_rtt"].unique():
                if rtt_ == 10:
                    marker = "o"
                elif rtt_ == 20:
                    marker = "v"
                elif rtt_ == 30:
                    marker = "s"
                elif rtt_ == 40:
                    marker = "P"
                sns.stripplot(ax=axs, data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", legend=False, dodge=True, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
            continue


        df_cubicfq = df[df["cca_generic"] == "cubicfq"]
        df_cubicfq = df_cubicfq[df_cubicfq["kernel"] == "kernel6-1"]
        df_cubicfq = df_cubicfq.sort_values(by=['slice_perc', "delay_rtt"])

        extra_legend_patches = []

        for rtt_ in df_cubicfq["delay_rtt"].unique():
            if rtt_ == 10:
                marker = "o"
            elif rtt_ == 20:
                marker = "v"
            elif rtt_ == 30:
                marker = "s"
            elif rtt_ == 40:
                marker = "P"
            extra_legend_patches.append(Line2D([0], [0], linestyle='none', mfc=paletti[2], markersize=3,mec=paletti[2], marker=marker, label=f'{int(rtt_)}ms'))
            data_ = df_cubicfq[["slice_perc", "mbps","deadline_run", "delay_rtt"]].groupby(["slice_perc", "deadline_run", "delay_rtt"],as_index=False).median()
            sns.stripplot(ax=axs, data=data_[data_["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", jitter=False, alpha=0.6, palette=paletti, zorder=0,dodge=True, legend=True if rtt_ == 10 else False, marker = marker, size=3,linewidth=0.1)

        axs.set_xticks([0,1,2,3,4,5,6,7,8,9,10,11,12], ["10","","20","","30","","40","","50","","60","","70"], fontsize=plotconfig.FONT_SIZE-2)
        axs.tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-2)
        axs.vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5,11.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)
        axs.set_xlim(-0.5, 12.5)
        axs.set_title("CUBIC + fq",fontsize=plotconfig.FONT_SIZE-2,pad=3)
        axs.set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-2)

        axs.set_xlabel("")
        han = [
            Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.1, marker="o", markersize=3, label='fq pacing'),
            Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.6, marker="o", markersize=1.8, label='no pacing'),
        ]
        leg1 = axs.legend(handles = han,loc="lower right", bbox_to_anchor=(1.3, 0.6), handlelength=1, framealpha=0.4,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
        axs.add_artist(leg1)
        h, l = axs.get_legend_handles_labels()
        for enu,hii in enumerate(h):
            hii.set_alpha(1)
            l[enu] = f"{int(int(l[enu])/1000000)}ms"
        leg2 = axs.legend(handles=h, labels= l, title="timeslice", loc="lower right", handlelength=1.7, bbox_to_anchor=(1.3,0.3), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)
        axs.add_artist(leg2)
        axs.legend(handles=extra_legend_patches, title="RTT", loc="lower right", handlelength=1.7, bbox_to_anchor=(1.3, 0), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)


    axs.set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-2)
    axs.set_ylim(-float(args["rate"][0])*0.05,int(args["rate"][0]))
    
    if savefig:
        fig.savefig(f"figures/figure_13.pdf", format="pdf")
    else:
        plt.show()

<>:55: SyntaxWarning: invalid escape sequence '\%'
<>:55: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_242551/1147250772.py:55: SyntaxWarning: invalid escape sequence '\%'
  axs.set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-2)


In [6]:
strip(df, True)

/tmp/ipykernel_242551/1147250772.py:27: UserWarning: The palette list has more values (3) than needed (2), which may not be intended.
  sns.stripplot(ax=axs, data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", legend=False, dodge=True, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
/tmp/ipykernel_242551/1147250772.py:27: UserWarning: The palette list has more values (3) than needed (2), which may not be intended.
  sns.stripplot(ax=axs, data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", legend=False, dodge=True, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
